In [ ]:
from pathlib import Path
import jax.numpy as jnp
from emergex import *

This is the setup of a reaction network

In [ ]:
# Units are arbitrary, so long as they are INTERNALLY CONSISTENT
# (i.e., if your second-order rates use 1/sec/uM, all concentrations should be in uM)

simTime = 600 # In units of sec
simResolution = 201 # Number of points retrieved

rxn1 = Reaction(5e-5, "A + B -> C") # In units of 1/sec/uM
rxn2 = Reaction(1e-2, "C + D <-> CD", backwardRate = 1e-0) # In units of 1/sec/uM, and in units of 1/sec
rxn3 = Reaction(8e-2, "CD -> D + E") # In units of 1/sec

badRxn1 = Reaction(5e-5 / 10, "A + B -> C") # In units of 1/sec/uM
badRxn2 = Reaction(1e-2 * 7, "C + D <-> CD", backwardRate = 1e-0) # In units of 1/sec/uM, and in units of 1/sec
badRxn3 = Reaction(8e-2 / 100, "CD -> D + E") # In units of 1/sec

compA = Component("A", 100) # In units of uM
compB = Component("B", 80) # In units of uM
compD2_5 = Component("D", 2.5) # In units of uM
compD5 = Component("D", 5) # In units of uM
compD7_5 = Component("D", 7.5) # In units of uM
compD10 = Component("D", 10) # In units of uM

d2_5Net = ReactionNetwork()
d5Net = ReactionNetwork()
d7_5Net = ReactionNetwork()
d10Net = ReactionNetwork()
testingNetwork = ReactionNetwork()

d2_5Net.addRxns([rxn1, rxn2, rxn3])
d5Net.addRxns([rxn1, rxn2, rxn3])
d7_5Net.addRxns([rxn1, rxn2, rxn3])
d10Net.addRxns([rxn1, rxn2, rxn3])

d2_5Net.addComponents([compA, compB, compD2_5])
d5Net.addComponents([compA, compB, compD5])
d7_5Net.addComponents([compA, compB, compD7_5])
d10Net.addComponents([compA, compB, compD10])

testingNetwork.addRxns([badRxn1, badRxn2, badRxn3])
testingNetwork.addComponents([compA, compB, compD5])

d2_5Net.simulateReactionFn(simTime, simDataResolution = simResolution)
d5Net.simulateReactionFn(simTime, simDataResolution = simResolution)
d7_5Net.simulateReactionFn(simTime, simDataResolution = simResolution)
d10Net.simulateReactionFn(simTime, simDataResolution = simResolution)

expNets = [d2_5Net, d5Net, d7_5Net, d10Net]

components = ["C","CD", "E"] # Add or remove what you would like plotted

After reaction network has been simulated, the simulated c and e component data is converted to jax array...

In [ ]:
def getComponentArrays(expNets: list[ReactionNetwork], comp:str) -> list[jnp.ndarray]:
    compArrayList:list[jnp.ndarray] = []
    for i in expNets:
        compArrayList.append(jnp.array(i._SimResults[0][comp]))
    return compArrayList

In [ ]:
def get2ComponentArrays(expNets: list[ReactionNetwork], comp1:str, comp2:str) -> list[jnp.ndarray]:
    compsArrayList:list[jnp.ndarray] = []
    for i in expNets:
        array1 = jnp.array(i._SimResults[0][comp1])
        array2 = jnp.array(i._SimResults[0][comp2])
        compsArrayList.append(array1 + array2)
        
    return compsArrayList

In [ ]:
eResults = getComponentArrays(expNets, "E")
cCompResults = get2ComponentArrays(expNets, "C", "CD")

timeArray:jnp.ndarray = jnp.array(d2_5Net._SimResults[0]["Time"])

...  so some random noise can be added before sending it into optimization software.

In [ ]:
import jax.numpy as jnp
import jax.random as random

def addRandomNoiseAndNormalizeArray(array:jnp.ndarray, normVal:float, scalar = 0.1, mult = True):
    # Create a random key
    key = random.PRNGKey(0)

    # Generate gaussian noise and multiply
    if (mult):
        array = array * (1 + random.normal(key, shape=array.shape) * scalar)
    else :
        array = array + (random.normal(key, shape=array.shape) * scalar)
        
    # Normalize by max value
    array = array / normVal
    return array

In [ ]:
for i in range(len(eResults)):
    eResults[i] = addRandomNoiseAndNormalizeArray(eResults[i], 80, 2, mult=False)
for i in range(len(cCompResults)):
    cCompResults[i] = addRandomNoiseAndNormalizeArray(cCompResults[i], 80, 2, mult=False)


Now, we create the experiment manager object, which is used as a container of multiple experiments with the same time spans. The IRL equivelent would be a plate in a plate reader. 

In [ ]:
exptGroupCompiler = ExperimentGroupCompiler([timeArray], testingNetwork)

def determineMaxDownstream(conc):
    return jnp.minimum(conc["A"], conc["B"])

dConcentrations = [
    2.5, 5.0, 7.5, 10.0
]

for i in range(len(dConcentrations)):

    interruptions = [Interruption(compD5, dConcentrations[i])]

    timeSpan = exptGroupCompiler.createTimeSpanByBreakInterval(interruptions = interruptions)

    expMaker = exptGroupCompiler.addExperiment([timeSpan])

    signal1 = Signal(
        ["C", "CD"],
        maxSignalConc=determineMaxDownstream,
        dataPoints=cCompResults[i])

    signal2 = Signal(
        "E",
        maxSignalConc=determineMaxDownstream,
        dataPoints=eResults[i])

    expMaker.addSignal(signal1)
    expMaker.addSignal(signal2)


exptManager = OptimizeExperimentsManager([exptGroupCompiler.convertToExperimentGroup()])

Next is the simulation runner object. This can have additional information for minimal rates and other simulation running data, but we can just use the defaults.

In [ ]:
simulationRunnerObj = CRNSimulationRunner()

Here is the CRNOptimizationFramework, which controls which variables can be modified. Here, just the reaction rates.

In [ ]:
freeParams = [
    FreeParameter(badRxn1),
    FreeParameter(badRxn2),
    FreeParameter(badRxn2, "BackwardRate"),
    FreeParameter(badRxn3),
]

optimizerFramework = CRNOptimizationFramework(freeParams)

After we have all the components, can throw this into a data result and save it.

In [ ]:
dataResult = CompiledOptimizationData(exptManager, simulationRunnerObj, optimizerFramework, iterationCount = 1000, callbackFrequency = 1)
dataResult.save(Path("."), "script2Pickel")

Now, we can load the object we made at any time and visualize it.

In [ ]:
dataResult = CompiledOptimizationData.load("./script2Pickel.pkl")
print(dataResult.OptimizerRunResultObject)

In [ ]:
iterationList1 = []
for i in range(1, 71, 10):
    iterationList1.append(i)
    
iterationList1.append(71)

saveExperimentResultsVisualization(
    dataResult,
    iterationList = iterationList1, #jnp.arange(1, 200, 4),
    fileName = "experimentFit_v31",
    fps = 2,
    timeUnits = "min"
)